In [ ]:
# Check for Duplicates in 'Lien' Column
import pandas as pd
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
df = pd.read_csv('../csv/STEP01_maisons_dept29.csv', encoding='utf-8')
display("Number of duplicated links:", df['Lien'].duplicated().sum())
duplicates = df[df['Lien'].duplicated(keep=False)].sort_values(by='Lien')
display(duplicates)

In [ ]:
# Extraire toutes les villes du département 29 depuis code_insee.csv et créer un nouveau CSV
import pandas as pd

# Charger le fichier code_insee.csv
df_insee = pd.read_csv('../csv/code_insee.csv', encoding='utf-8')

# Filtrer pour le département 35
df_dept35 = df_insee[df_insee['DEP'] == '35']

# Sauvegarder dans un nouveau CSV
df_dept35.to_csv('../csv/communes_dept35.csv', index=False, encoding='utf-8')

print(f"Nombre de communes dans le département 35: {len(df_dept35)}")
print("Nouveau CSV créé: ../csv/communes_dept35.csv")

## Outliers par région

In [ ]:
import pandas as pd
import numpy as np

# Load CSV
df = pd.read_csv('../csv/STEP02_maisons_dept29.csv', encoding='utf-8')

# Nettoyage numérique
df['Taille'] = pd.to_numeric(df['Taille'].astype(str).str.replace(' ', ''), errors='coerce')

# Nettoyage du prix
df['Prix'] = pd.to_numeric(df['Prix'].astype(str).str.replace(' ', ''), errors='coerce')

# Prix par m² is already calculated and rounded in the CSV as 'Prix au m2'

# --- Méthode 1: Détection outliers avec moyenne ± 2 écarts-types ---
mean_pm2 = df['Prix au m2'].mean()
std_pm2 = df['Prix au m2'].std()

print(f"Moyenne du prix au m²: {mean_pm2:.2f} €")
print(f"Écart-type: {std_pm2:.2f} €")

seuil_haut = mean_pm2 + 2 * std_pm2
seuil_bas = mean_pm2 - 2 * std_pm2

outliers_mean_std = df[(df['Prix au m2'] > seuil_haut) | (df['Prix au m2'] < seuil_bas)]
print(f"\nOutliers selon moyenne ± 2 écarts-types ({len(outliers_mean_std)} au total):")
print(outliers_mean_std[['Prix au m2', 'Lieu', 'Prix', 'Taille']].to_string(index=False))

# Moyenne sans outliers méthode 1
df_sans_outliers_1 = df[~df.index.isin(outliers_mean_std.index)]
mean_sans_1 = df_sans_outliers_1['Prix au m2'].mean()
print(f"Moyenne sans outliers (méthode 1): {mean_sans_1:.2f} €")

# --- Méthode 2: Détection outliers avec IQR ---
Q1 = df['Prix au m2'].quantile(0.25)
Q3 = df['Prix au m2'].quantile(0.75)
IQR = Q3 - Q1

print(f"\nQ1: {Q1:.2f}, Q3: {Q3:.2f}, IQR: {IQR:.2f}")

lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

outliers_iqr = df[(df['Prix au m2'] < lower) | (df['Prix au m2'] > upper)]
print(f"\nOutliers selon IQR ({len(outliers_iqr)} au total):")
print(outliers_iqr[['Prix au m2', 'Lieu', 'Prix', 'Taille']].to_string(index=False))

# Moyenne sans outliers méthode 2
df_sans_outliers_2 = df[~df.index.isin(outliers_iqr.index)]
mean_sans_2 = df_sans_outliers_2['Prix au m2'].mean()
print(f"Moyenne sans outliers (méthode 2): {mean_sans_2:.2f} €")

# Maison avec le plus gros prix au m² (souvent un outlier)
maison_max = df.loc[df['Prix au m2'].idxmax()]
print("\nMaison avec le plus gros prix au m² :")
print(maison_max)